# POS delay - confronto LSTM vs Autoencoder reconstruction-based

Il notebook confronta due famiglie di approcci per la rilevazione dei POS delay:

- **LSTM forecasting-based**, già addestrata su `pos_net_cf`, valutata con il detector POS a profili;
- **LSTM Autoencoder reconstruction-based**, valutato tramite errore di ricostruzione della finestra.

La differenza metodologica è intenzionale: la LSTM produce una previsione puntuale usata dal detector a profili, mentre l'Autoencoder produce direttamente uno score di anomalia basato su reconstruction error. Per l'AE, quindi, la rilevazione non passa più da `pos_cos`, `pos_l1` o `pos_wasserstein`, ma da:

```text
finestra POS reale -> AE -> finestra ricostruita -> ae_mae_score -> z-score validation -> finestre rilevate
```

Lo score principale dell'AE è `ae_mae_score`, coerente con la loss MAE usata in training.

## Path del progetto

La root della repository viene individuata risalendo dalla cartella di lavoro; i path degli artifact sono definiti in `project_paths.py`.

In [1]:
# =========================================================
# PATH
# =========================================================

import sys
from pathlib import Path

start_dir = Path.cwd().resolve()

for candidate in [start_dir, *start_dir.parents]:
    if (candidate / "project_paths.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Root del progetto non trovata. "
        "Avvia Jupyter da una cartella interna alla repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/Users/ciok4/jupyter file/tesi')

## Import

Librerie per preprocessing, training Keras, valutazione event-level e salvataggio dei risultati.

In [2]:
import pickle
import time
from itertools import product

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Embedding,
    Concatenate,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm
from IPython.display import display

from lstm_utils import (
    build_pos_ae_inputs,
    create_pos_ae_windows,
    evaluate_detected_windows_event_level,
    log_transform,
)

from POS_delay_utils import (
    POS_DELAY_DETECTOR_CONFIG,
    add_detection_offsets,
    build_gt_pos_delay_windows,
    list_pos_delay_sensitivity_datasets,
    pooled_f1,
    pooled_precision,
    pooled_recall,
    prepare_pos_dataframe,
)

from project_paths import (
    CLEAN_DATA_PATH,
    POS_DELAY_RESULTS_DIR,
    POS_DELAY_SENSITIVITY_DIR,
    POS_MODEL_DIR,
    ensure_artifact_directories,
)

pd.set_option("display.max_columns", None)

## Configurazione dell'esperimento

La griglia riguarda solo l'Autoencoder (`window_size × latent_dim`). Per l'AE il detector usa `ae_mae_score`; i risultati LSTM sono invece caricati dalla sensitivity analysis con detector a profili.

In [3]:
ensure_artifact_directories()

BASE_SENSITIVITY_PATH = POS_DELAY_SENSITIVITY_DIR
LSTM_RESULTS_PATH = (
    POS_DELAY_RESULTS_DIR
    / "lstm_sensitivity"
    / "pos_delay_sensitivity_best_detector_results.csv"
)

AE_MODEL_ROOT = POS_MODEL_DIR / "ae_reconstruction"
OUTPUT_DIR = POS_DELAY_RESULTS_DIR / "ae_vs_lstm_comparison"
AE_INFERENCE_CACHE_DIR = OUTPUT_DIR / "_ae_inference_cache"
TABLES_DIR = OUTPUT_DIR / "tables"

for path in [AE_MODEL_ROOT, OUTPUT_DIR, AE_INFERENCE_CACHE_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Griglia del modello AE.
AE_WINDOW_VALUES = [5, 7, 10]
LATENT_DIM_VALUES = [2, 4, 8, 16]

# Detector AE reconstruction-based.
AE_SCORE_COL = "ae_mae_score"
AE_Z_THRESHOLD = 3.5
AE_MIN_CONSECUTIVE = 2
AE_GAP_TOLERANCE = 1
AE_IOU_THRESHOLD = 0.20

# Configurazione del detector LSTM già applicata nel notebook 02.
LSTM_DETECTOR_CONFIG = POS_DELAY_DETECTOR_CONFIG.copy()

# Parametri training AE.
AE_LSTM_UNITS = 64
AE_LEARNING_RATE = 0.001
AE_EPOCHS = 100
AE_BATCH_SIZE = 32
AE_PATIENCE = 20

TRAIN_SIZE = 0.70
VAL_SIZE = 0.10

# Esperimento principale.
SOURCE_DURATION_FILTER = [1]

DELAY_TYPES_FILTER = [
    "mild_delay",
    "moderate_delay",
    "strong_delay",
    "batch_backlog",
    "settlement_freeze",
]

# Cache e ricalcolo.
# Il CSV raw evita di rieseguire l'intera griglia AE quando i risultati sono completi.
FORCE_RETRAIN_AE = False
FORCE_RECOMPUTE_AE_INFERENCE = False
FORCE_RECOMPUTE_RESULTS = False

GLOBAL_SEED = 42

AE_RAW_RESULTS_PATH = TABLES_DIR / "ae_reconstruction_grid_raw_results.csv"
AE_GT_EVAL_PATH = TABLES_DIR / "ae_reconstruction_gt_eval.csv"
AE_DET_EVAL_PATH = TABLES_DIR / "ae_reconstruction_det_eval.csv"
AE_GRID_OVERALL_PATH = TABLES_DIR / "ae_reconstruction_grid_overall_results.csv"
AE_FAMILY_PATH = TABLES_DIR / "ae_reconstruction_family_by_window_latent.csv"
COMPACT_TABLE_PATH = TABLES_DIR / "compact_lstm_vs_all_ae_configs.csv"

CONFIG = {
    "ae_window_values": AE_WINDOW_VALUES,
    "latent_dim_values": LATENT_DIM_VALUES,
    "ae_score_col": AE_SCORE_COL,
    "ae_z_threshold": AE_Z_THRESHOLD,
    "ae_min_consecutive": AE_MIN_CONSECUTIVE,
    "ae_gap_tolerance": AE_GAP_TOLERANCE,
    "lstm_detector_config": LSTM_DETECTOR_CONFIG,
    "source_duration_filter": SOURCE_DURATION_FILTER,
    "force_retrain_ae": FORCE_RETRAIN_AE,
    "force_recompute_ae_inference": FORCE_RECOMPUTE_AE_INFERENCE,
    "force_recompute_results": FORCE_RECOMPUTE_RESULTS,
}

CONFIG

{'ae_window_values': [5, 7, 10],
 'latent_dim_values': [2, 4, 8, 16],
 'ae_score_col': 'ae_mae_score',
 'ae_z_threshold': 3.5,
 'ae_min_consecutive': 2,
 'ae_gap_tolerance': 1,
 'lstm_detector_config': {'score_col': 'pos_cos',
  'profile_window_size': 7,
  'z_threshold': 3.5,
  'min_consecutive': 2,
  'gap_tolerance': 1,
  'detected_window_mode': 'profile_windows_union',
  'iou_threshold': 0.2},
 'source_duration_filter': [1],
 'force_retrain_ae': False,
 'force_recompute_ae_inference': False,
 'force_recompute_results': False}

## Riproducibilità

In [4]:
def set_global_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_global_seed(GLOBAL_SEED)

## Utility di aggregazione

Le metriche finali sono event-level. Le aggregazioni usano conteggi pooled (`tp`, `fp`, `fn`) e, in parallelo, medie tra run.

In [5]:
def aggregate_results(df, group_cols):
    """Aggrega metriche event-level con conteggi pooled."""
    agg = (
        df
        .groupby(group_cols, dropna=False, as_index=False)
        .agg(
            n_runs=("seed", "count"),
            n_gt_events=("n_gt_events", "sum"),
            n_detected_events=("n_detected_events", "sum"),
            tp=("tp", "sum"),
            fp=("fp", "sum"),
            fn=("fn", "sum"),
            precision_mean=("precision", "mean"),
            precision_std=("precision", "std"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            mean_iou_mean=("mean_iou", "mean"),
            mean_iou_std=("mean_iou", "std"),
            det_offset_start_mean=("mean_det_offset_start", "mean"),
            det_offset_start_std=("mean_det_offset_start", "std"),
            det_offset_end_mean=("mean_det_offset_end", "mean"),
            det_offset_end_std=("mean_det_offset_end", "std"),
        )
    )

    agg["precision_pooled"] = [
        pooled_precision(tp, fp)
        for tp, fp in zip(agg["tp"], agg["fp"])
    ]

    agg["recall_pooled"] = [
        pooled_recall(tp, fn)
        for tp, fn in zip(agg["tp"], agg["fn"])
    ]

    agg["f1_pooled"] = [
        pooled_f1(p, r)
        for p, r in zip(agg["precision_pooled"], agg["recall_pooled"])
    ]

    return agg


def split_daily_df(df, split="test", train_size=0.70, val_size=0.10):
    """Estrae train/val/test giornaliero con split temporale per store."""
    temp = df.copy()
    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp.sort_values(["store_id", "date"])

    parts = []

    for store_id, g in temp.groupby("store_id"):
        g = g.sort_values("date").copy()

        n = len(g)
        train_end = int(train_size * n)
        val_end = int((train_size + val_size) * n)

        if split == "train":
            part = g.iloc[:train_end].copy()
        elif split == "val":
            part = g.iloc[train_end:val_end].copy()
        elif split == "test":
            part = g.iloc[val_end:].copy()
        else:
            raise ValueError("split must be 'train', 'val' or 'test'")

        parts.append(part)

    return pd.concat(parts, ignore_index=True)

## Feature usate dall'Autoencoder

L'AE usa feature POS coerenti con la LSTM, ma la sua uscita è una finestra ricostruita di `pos_net_cf`. Le ground truth sono conservate solo per valutazione.

In [6]:
POS_GT_COLS = [
    "is_point_anomaly",
    "pa_type",
    "pa_mult",
    "is_pos_delay_source_day",
    "pos_delay_source_day_in_event",
    "pos_delay_source_duration",
    "is_pos_delay_effect_day",
    "pos_delay_effect_day_in_event",
    "pos_delay_effect_duration",
    "pos_delay_event_id",
    "pos_delay_type",
]


def get_pos_ae_feature_lists():
    seq_num_features = [
        "pos_card_sales",
        "pos_card_sales_rm_30",
    ]

    seq_bool_features = [
        "holiday",
        "pre_holiday",
    ]

    seq_cat_features = [
        "week_day",
        "month",
        "store_id",
    ]

    target = "pos_net_cf"

    log_cols = [
        "pos_net_cf",
        "pos_card_sales",
        "pos_card_sales_rm_30",
    ]

    return {
        "seq_num": seq_num_features,
        "seq_bool": seq_bool_features,
        "seq_cat": seq_cat_features,
        "target": target,
        "ground_truth": POS_GT_COLS,
        "log_transform": log_cols,
        "scale": list(dict.fromkeys(seq_num_features + [target])),
    }


def ensure_pos_gt_columns(df):
    """Aggiunge colonne GT mancanti con valori normali."""
    df = df.copy()

    defaults = {
        "is_point_anomaly": 0,
        "pa_type": "normal",
        "pa_mult": 1.0,
        "is_pos_delay_source_day": 0,
        "pos_delay_source_day_in_event": -1,
        "pos_delay_source_duration": 0,
        "is_pos_delay_effect_day": 0,
        "pos_delay_effect_day_in_event": -1,
        "pos_delay_effect_duration": 0,
        "pos_delay_event_id": -1,
        "pos_delay_type": "normal",
    }

    for col, value in defaults.items():
        if col not in df.columns:
            df[col] = value

    return df


def prepare_pos_ae_dataframe(df, features, mappings=None, fit_mappings=True):
    """Preprocessing comune per training, inference e costruzione GT encoded."""
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["store_id", "date"]).copy()
    df = ensure_pos_gt_columns(df)

    required_cols = (
        ["date", "store_id", "actual_holiday", "holiday", "week_day", "month"]
        + ["pos_card_sales", features["target"]]
        + features["ground_truth"]
    )

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Mancano colonne richieste per POS AE: {missing_cols}")

    df["pre_holiday"] = (
        df.groupby("store_id")["actual_holiday"]
          .shift(-1)
          .fillna(0)
          .astype(int)
    )

    df["pos_card_sales_rm_30"] = (
        df.groupby("store_id")["pos_card_sales"]
          .transform(lambda s: s.shift(1).rolling(30, min_periods=1).mean())
    )

    df["pos_card_sales_rm_30"] = (
        df.groupby("store_id")["pos_card_sales_rm_30"]
          .bfill()
    )

    df = log_transform(df, features["log_transform"])

    if fit_mappings:
        mappings = {}
        for col in features["seq_cat"]:
            df[col], mapping = pd.factorize(df[col])
            mappings[col] = mapping
    else:
        if mappings is None:
            raise ValueError("mappings must be provided when fit_mappings=False")

        for col, mapping in mappings.items():
            df[col] = pd.Categorical(df[col], categories=mapping).codes
            if (df[col] < 0).any():
                bad_values = df.loc[df[col] < 0, col].unique()
                raise ValueError(
                    f"Valori non presenti nel mapping salvato per colonna {col}: {bad_values}"
                )

    return df, mappings

## Costruzione dataset AE

Split temporale per store, trasformazioni logaritmiche e scaling stimato solo sul training set. Lo score AE viene calcolato nella stessa scala della loss, quindi su target trasformato e standardizzato.

In [7]:
def concat_parts(parts):
    return {
        key: np.concatenate([p[key] for p in parts], axis=0)
        for key in parts[0].keys()
    }


def build_pos_ae_dataset_train(df, window_size=7, train_size=0.70, val_size=0.10):
    """Costruisce train/validation/test per il training AE."""
    features = get_pos_ae_feature_lists()
    df, mappings = prepare_pos_ae_dataframe(
        df,
        features,
        mappings=None,
        fit_mappings=True,
    )

    train_parts = []
    val_parts = []
    test_parts = []
    feature_scalers = {}

    for store_id in df["store_id"].unique():
        temp = df[df["store_id"] == store_id].sort_values("date").copy()

        n = len(temp)
        train_end = int(train_size * n)
        val_end = int((train_size + val_size) * n)

        train_df = temp.iloc[:train_end].copy()
        val_df = temp.iloc[train_end:val_end].copy()
        test_df = temp.iloc[val_end:].copy()

        scaler = StandardScaler()
        scaler.fit(train_df[features["scale"]].astype(float))
        feature_scalers[store_id] = scaler

        train_df[features["scale"]] = scaler.transform(train_df[features["scale"]].astype(float))
        val_df[features["scale"]] = scaler.transform(val_df[features["scale"]].astype(float))
        test_df[features["scale"]] = scaler.transform(test_df[features["scale"]].astype(float))

        train_parts.append(create_pos_ae_windows(train_df, features, window_size))
        val_parts.append(create_pos_ae_windows(val_df, features, window_size))
        test_parts.append(create_pos_ae_windows(test_df, features, window_size))

    return (
        concat_parts(train_parts),
        concat_parts(val_parts),
        concat_parts(test_parts),
        feature_scalers,
        mappings,
        features,
    )


def build_pos_ae_dataset_inference(
    df,
    feature_scalers,
    mappings,
    features,
    window_size=7,
    train_size=0.70,
    val_size=0.10,
):
    """Costruisce gli split di inference usando scaler e mapping salvati."""
    features = features.copy()
    features["ground_truth"] = POS_GT_COLS

    df_encoded, _ = prepare_pos_ae_dataframe(
        df,
        features,
        mappings=mappings,
        fit_mappings=False,
    )

    train_parts = []
    val_parts = []
    test_parts = []

    for store_id in df_encoded["store_id"].unique():
        temp = df_encoded[df_encoded["store_id"] == store_id].sort_values("date").copy()

        n = len(temp)
        train_end = int(train_size * n)
        val_end = int((train_size + val_size) * n)

        train_df = temp.iloc[:train_end].copy()
        val_df = temp.iloc[train_end:val_end].copy()
        test_df = temp.iloc[val_end:].copy()

        scaler = feature_scalers[store_id]

        train_df[features["scale"]] = scaler.transform(train_df[features["scale"]].astype(float))
        val_df[features["scale"]] = scaler.transform(val_df[features["scale"]].astype(float))
        test_df[features["scale"]] = scaler.transform(test_df[features["scale"]].astype(float))

        train_parts.append(create_pos_ae_windows(train_df, features, window_size))
        val_parts.append(create_pos_ae_windows(val_df, features, window_size))
        test_parts.append(create_pos_ae_windows(test_df, features, window_size))

    return (
        concat_parts(train_parts),
        concat_parts(val_parts),
        concat_parts(test_parts),
        df_encoded,
    )

## Modello LSTM Autoencoder

Encoder LSTM, bottleneck e decoder LSTM. `latent_dim` controlla la capacità del bottleneck: se è troppo alto, l'AE può ricostruire anche finestre anomale e ridurre lo score di anomalia.

In [8]:
def build_pos_lstm_autoencoder(train, latent_dim=8, lstm_units=64, learning_rate=0.001):
    X_seq_num = train["X_seq_num"]
    X_seq_bool = train["X_seq_bool"]
    X_seq_cat = train["X_seq_cat"]

    window_size = X_seq_num.shape[1]
    n_num = X_seq_num.shape[2]
    n_bool = X_seq_bool.shape[2]

    n_weekday = int(X_seq_cat[:, :, 0].max()) + 1
    n_month = int(X_seq_cat[:, :, 1].max()) + 1
    n_store = int(X_seq_cat[:, :, 2].max()) + 1

    seq_num_input = Input(shape=(window_size, n_num), name="seq_num_input")
    seq_bool_input = Input(shape=(window_size, n_bool), name="seq_bool_input")
    weekday_input = Input(shape=(window_size,), name="weekday_input")
    month_input = Input(shape=(window_size,), name="month_input")
    store_input = Input(shape=(window_size,), name="store_input")

    weekday_emb = Embedding(input_dim=n_weekday, output_dim=3, name="weekday_embedding")(weekday_input)
    month_emb = Embedding(input_dim=n_month, output_dim=3, name="month_embedding")(month_input)
    store_emb = Embedding(input_dim=n_store, output_dim=4, name="store_embedding")(store_input)

    x = Concatenate(axis=-1, name="ae_input_concat")([
        seq_num_input,
        seq_bool_input,
        weekday_emb,
        month_emb,
        store_emb,
    ])

    x = LSTM(lstm_units, return_sequences=True, name="encoder_lstm_1")(x)
    latent = LSTM(latent_dim, return_sequences=False, name="latent")(x)
    x = RepeatVector(window_size, name="repeat_latent")(latent)
    x = LSTM(lstm_units, return_sequences=True, name="decoder_lstm")(x)

    output = TimeDistributed(Dense(1), name="pos_net_cf_reconstruction")(x)

    model = Model(
        inputs=[seq_num_input, seq_bool_input, weekday_input, month_input, store_input],
        outputs=output,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mae",
        metrics=["mae"],
    )

    return model

## Training e caricamento degli AE

Ogni configurazione della griglia salva modello e artifact di preprocessing. In assenza di `FORCE_RETRAIN_AE`, gli artifact completi vengono riutilizzati.

In [9]:
def get_ae_model_dir(window_size, latent_dim):
    return AE_MODEL_ROOT / f"w{window_size}_latent{latent_dim}"


def load_train_dataframe_for_ae():
    if not CLEAN_DATA_PATH.exists():
        raise FileNotFoundError(
            f"Dataset clean non trovato: {CLEAN_DATA_PATH}"
        )

    print(f"Uso dataset clean per training AE: {CLEAN_DATA_PATH}")

    df_train = pd.read_csv(CLEAN_DATA_PATH)
    df_train["date"] = pd.to_datetime(df_train["date"])
    df_train = df_train.sort_values(["store_id", "date"]).reset_index(drop=True)

    return ensure_pos_gt_columns(df_train)


def train_or_load_ae(window_size, latent_dim):
    model_dir = get_ae_model_dir(window_size, latent_dim)
    model_path = model_dir / "lstm_pos_ae.keras"
    scalers_path = model_dir / "feature_scalers.pkl"
    mappings_path = model_dir / "mappings.pkl"
    features_path = model_dir / "features.pkl"
    history_path = model_dir / "history.pkl"

    if (
        model_path.exists()
        and scalers_path.exists()
        and mappings_path.exists()
        and features_path.exists()
        and not FORCE_RETRAIN_AE
    ):
        model = tf.keras.models.load_model(model_path, safe_mode=False)

        with open(scalers_path, "rb") as f:
            feature_scalers = pickle.load(f)

        with open(mappings_path, "rb") as f:
            mappings = pickle.load(f)

        with open(features_path, "rb") as f:
            features = pickle.load(f)

        history = None
        if history_path.exists():
            with open(history_path, "rb") as f:
                history = pickle.load(f)

        return {
            "model": model,
            "feature_scalers": feature_scalers,
            "mappings": mappings,
            "features": features,
            "history": history,
            "model_dir": model_dir,
            "loaded_from_disk": True,
        }

    model_dir.mkdir(parents=True, exist_ok=True)

    print(f"Training AE: window_size={window_size}, latent_dim={latent_dim}")

    df_train = load_train_dataframe_for_ae()

    train, val, test, feature_scalers, mappings, features = build_pos_ae_dataset_train(
        df_train,
        window_size=window_size,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
    )

    train_inputs = build_pos_ae_inputs(train)
    val_inputs = build_pos_ae_inputs(val)

    tf.keras.backend.clear_session()
    set_global_seed(GLOBAL_SEED)

    model = build_pos_lstm_autoencoder(
        train,
        latent_dim=latent_dim,
        lstm_units=AE_LSTM_UNITS,
        learning_rate=AE_LEARNING_RATE,
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=AE_PATIENCE,
        restore_best_weights=True,
    )

    history_obj = model.fit(
        train_inputs,
        train["y"],
        validation_data=(val_inputs, val["y"]),
        epochs=AE_EPOCHS,
        batch_size=AE_BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1,
    )

    history = history_obj.history

    model.save(model_path)

    with open(scalers_path, "wb") as f:
        pickle.dump(feature_scalers, f)

    with open(mappings_path, "wb") as f:
        pickle.dump(mappings, f)

    with open(features_path, "wb") as f:
        pickle.dump(features, f)

    with open(history_path, "wb") as f:
        pickle.dump(history, f)

    return {
        "model": model,
        "feature_scalers": feature_scalers,
        "mappings": mappings,
        "features": features,
        "history": history,
        "model_dir": model_dir,
        "loaded_from_disk": False,
    }

## Score reconstruction-based dell'AE

Ogni finestra produce direttamente score di ricostruzione. Lo score principale è `ae_mae_score`, calcolato sull'intera finestra nella scala modello. Gli altri score sono salvati solo per diagnostica.

In [10]:
def make_pos_ae_reconstruction_results_df(data, y_pred, gt_cols=POS_GT_COLS):
    """
    Converte le ricostruzioni AE in finestre reconstruction-based.

    Una riga = una finestra ricostruita.
    La detection AE userà direttamente ae_mae_score, non il detector a profili.
    """
    y_true = np.asarray(data["y"], dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_pred.ndim == 2:
        y_pred = y_pred.reshape(y_true.shape)

    errors = y_true - y_pred
    abs_errors = np.abs(errors)

    window_size = y_true.shape[1]
    center_pos = window_size // 2
    half_left = center_pos
    half_right = window_size - center_pos - 1

    center_dates = pd.to_datetime(data["date"])

    results = pd.DataFrame({
        "store_id": data["store_id"].astype(int),
        "center_date": center_dates,
        "window_start": center_dates - pd.to_timedelta(half_left, unit="D"),
        "window_end": center_dates + pd.to_timedelta(half_right, unit="D"),
        "ae_window_size": window_size,
        "ae_mae_score": abs_errors.mean(axis=(1, 2)),
        "ae_mse_score": np.square(errors).mean(axis=(1, 2)),
        "ae_max_abs_error": abs_errors.max(axis=(1, 2)),
        "ae_signed_mean_error": errors.mean(axis=(1, 2)),
        "center_abs_error": abs_errors[:, center_pos, 0],
    })

    gt = np.asarray(data["ground_truth"], dtype=object)

    # Ground truth al centro della finestra.
    center_gt = pd.DataFrame(
        gt[:, center_pos, :],
        columns=[f"center_{col}" for col in gt_cols],
    )

    results = pd.concat(
        [results.reset_index(drop=True), center_gt.reset_index(drop=True)],
        axis=1,
    )

    # Informazione window-level utile per diagnostica.
    gt_df_long = []
    for i in range(len(gt)):
        w = pd.DataFrame(gt[i], columns=gt_cols)

        source_mask = w["is_pos_delay_source_day"].astype(int) == 1
        effect_mask = w["is_pos_delay_effect_day"].astype(int) == 1
        event_mask = w["pos_delay_event_id"].astype(int) != -1

        event_candidates = w.loc[
            (source_mask | effect_mask) & event_mask,
            ["pos_delay_event_id", "pos_delay_type"],
        ]

        if event_candidates.empty:
            event_id_window = -1
            type_window = "normal"
        else:
            event_id_window = int(event_candidates["pos_delay_event_id"].mode().iloc[0])
            type_window = event_candidates["pos_delay_type"].mode().iloc[0]

        gt_df_long.append({
            "is_pos_delay_source_window": int(source_mask.any()),
            "is_pos_delay_effect_window": int(effect_mask.any()),
            "pos_delay_event_id_window": event_id_window,
            "pos_delay_type_window": type_window,
            "n_source_days_in_window": int(source_mask.sum()),
            "n_effect_days_in_window": int(effect_mask.sum()),
        })

    results = pd.concat(
        [results, pd.DataFrame(gt_df_long)],
        axis=1,
    )

    return results


def compute_ae_reconstruction_zscore(
    val_windows,
    test_windows,
    score_col="ae_mae_score",
    store_col="store_id",
):
    """Calcola z-score store-specific dello score AE usando il validation split."""
    val = val_windows.copy()
    test = test_windows.copy()

    stats = (
        val
        .groupby(store_col)[score_col]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={
            "mean": "ae_score_mean",
            "std": "ae_score_std",
        })
    )

    val = val.merge(stats, on=store_col, how="left")
    test = test.merge(stats, on=store_col, how="left")

    val["ae_zscore"] = (
        (val[score_col] - val["ae_score_mean"])
        / (val["ae_score_std"] + 1e-12)
    )

    test["ae_zscore"] = (
        (test[score_col] - test["ae_score_mean"])
        / (test["ae_score_std"] + 1e-12)
    )

    return val, test


def detect_ae_reconstruction_windows(
    test_windows,
    z_threshold=3.5,
):
    """Marca come rilevate le finestre AE con z-score sopra soglia."""
    out = test_windows.copy()
    out["is_ae_detected_window"] = (out["ae_zscore"] > z_threshold).astype(int)
    return out

## Costruzione delle finestre rilevate AE

Le finestre AE oltre soglia vengono raggruppate per store. `min_consecutive` riduce falsi positivi isolati; `gap_tolerance` permette di fondere blocchi quasi contigui.

In [11]:
def build_detected_windows_from_ae_reconstruction_windows(
    ae_windows,
    detected_col="is_ae_detected_window",
    store_col="store_id",
    center_col="center_date",
    window_start_col="window_start",
    window_end_col="window_end",
    score_col="ae_mae_score",
    min_consecutive=2,
    gap_tolerance=1,
):
    """Costruisce detected windows unendo finestre AE rilevate."""
    temp = ae_windows.copy()

    required_cols = [
        store_col,
        detected_col,
        center_col,
        window_start_col,
        window_end_col,
        score_col,
        "ae_zscore",
    ]

    missing_cols = [col for col in required_cols if col not in temp.columns]
    if missing_cols:
        raise ValueError(f"Mancano colonne richieste: {missing_cols}")

    for col in [center_col, window_start_col, window_end_col]:
        temp[col] = pd.to_datetime(temp[col])

    out = []

    for store_id, g in temp.groupby(store_col):
        g = (
            g.sort_values(center_col)
             .reset_index(drop=True)
             .copy()
        )

        g["_detected_raw"] = g[detected_col].astype(int)
        detected = g["_detected_raw"].values.copy()

        if gap_tolerance > 0:
            n = len(detected)
            i = 0

            while i < n:
                if detected[i] == 1:
                    i += 1
                    continue

                gap_start = i

                while i < n and detected[i] == 0:
                    i += 1

                gap_end = i - 1
                gap_len = gap_end - gap_start + 1

                has_left_detection = gap_start > 0 and detected[gap_start - 1] == 1
                has_right_detection = i < n and detected[i] == 1

                if has_left_detection and has_right_detection and gap_len <= gap_tolerance:
                    detected[gap_start:gap_end + 1] = 1

        g["_detected_smooth"] = detected
        g["_block"] = (
            g["_detected_smooth"]
            .ne(g["_detected_smooth"].shift())
            .cumsum()
        )

        detected_runs = (
            g[g["_detected_smooth"] == 1]
            .groupby("_block")
        )

        for _, run in detected_runs:
            if len(run) < min_consecutive:
                continue

            detected_start = run[window_start_col].min()
            detected_end = run[window_end_col].max()

            out.append({
                store_col: store_id,
                "detected_start": detected_start,
                "detected_end": detected_end,
                "detected_duration_calendar": (detected_end - detected_start).days + 1,
                "n_ae_windows_in_event": len(run),
                "n_raw_detected_ae_windows": int(run["_detected_raw"].sum()),
                "n_filled_gap_ae_windows": int(run["_detected_smooth"].sum() - run["_detected_raw"].sum()),
                f"{score_col}_mean": run[score_col].mean(),
                f"{score_col}_max": run[score_col].max(),
                "ae_zscore_mean": run["ae_zscore"].mean(),
                "ae_zscore_max": run["ae_zscore"].max(),
                "detected_center_start": run[center_col].min(),
                "detected_center_end": run[center_col].max(),
            })

    return pd.DataFrame(out)


def evaluate_ae_reconstruction_detector(
    val_windows,
    test_windows,
    test_daily_df,
    score_col="ae_mae_score",
    z_threshold=3.5,
    min_consecutive=2,
    gap_tolerance=1,
    iou_threshold=0.20,
):
    """Valuta AE reconstruction-based contro le effect windows POS."""
    val_scored, test_scored = compute_ae_reconstruction_zscore(
        val_windows,
        test_windows,
        score_col=score_col,
    )

    test_detected = detect_ae_reconstruction_windows(
        test_scored,
        z_threshold=z_threshold,
    )

    detected_windows = build_detected_windows_from_ae_reconstruction_windows(
        test_detected,
        detected_col="is_ae_detected_window",
        score_col=score_col,
        min_consecutive=min_consecutive,
        gap_tolerance=gap_tolerance,
    )

    gt_windows = build_gt_pos_delay_windows(test_daily_df)

    gt_eval, det_eval, summary = evaluate_detected_windows_event_level(
        gt_windows=gt_windows,
        detected_windows=detected_windows,
        iou_threshold=iou_threshold,
    )

    gt_eval = add_detection_offsets(gt_eval, det_eval)

    if not gt_eval.empty and "matched" in gt_eval.columns:
        matched_mask = gt_eval["matched"].astype(int) == 1
        summary["mean_det_offset_start"] = gt_eval.loc[
            matched_mask,
            "det_offset_start",
        ].mean()
        summary["mean_det_offset_end"] = gt_eval.loc[
            matched_mask,
            "det_offset_end",
        ].mean()
    else:
        summary["mean_det_offset_start"] = np.nan
        summary["mean_det_offset_end"] = np.nan

    return {
        "summary": summary,
        "gt_eval": gt_eval,
        "det_eval": det_eval,
        "val_ae_windows": val_scored,
        "test_ae_windows": test_detected,
        "detected_windows": detected_windows,
        "gt_windows": gt_windows,
    }

## Dataset di sensitivity

L'analisi principale resta su `source_duration = 1`, cioè eventi generati da un singolo source day.

In [12]:
datasets_df = list_pos_delay_sensitivity_datasets(
    BASE_SENSITIVITY_PATH,
    source_duration_filter=SOURCE_DURATION_FILTER,
    delay_types_filter=DELAY_TYPES_FILTER,
)

datasets_df

,path,delay_type,source_duration,seed
0,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,42
1,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,43
2,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,44
3,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,45
4,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,batch_backlog,1,46
5,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,mild_delay,1,42
6,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,mild_delay,1,43
7,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,mild_delay,1,44
8,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,mild_delay,1,45
9,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,mild_delay,1,46


In [13]:
if not datasets_df.empty:
    display(
        datasets_df
        .groupby(["source_duration", "delay_type"])
        .size()
        .rename("n_datasets")
        .reset_index()
    )
else:
    print("Nessun dataset trovato. Controlla BASE_SENSITIVITY_PATH.")

,source_duration,delay_type,n_datasets
0,1,batch_backlog,5
1,1,mild_delay,5
2,1,moderate_delay,5
3,1,settlement_freeze,5
4,1,strong_delay,5


## Cache dei risultati AE reconstruction-based

La cache per dataset conserva finestre AE, score di ricostruzione e test giornaliero codificato. Il CSV raw della griglia evita inoltre di ripetere l'intera valutazione quando i risultati sono già completi.

In [14]:
def ae_dataset_cache_path(window_size, latent_dim, dataset_row):
    delay_type = dataset_row["delay_type"]
    source_duration = int(dataset_row["source_duration"])
    seed = int(dataset_row["seed"])

    cache_subdir = AE_INFERENCE_CACHE_DIR / f"w{window_size}_latent{latent_dim}"
    cache_subdir.mkdir(parents=True, exist_ok=True)

    return cache_subdir / f"{delay_type}_srcdur_{source_duration}_seed_{seed}.pkl"


def compute_or_load_ae_reconstruction_results(window_size, latent_dim, ae_payload, dataset_row):
    cache_path = ae_dataset_cache_path(window_size, latent_dim, dataset_row)

    if cache_path.exists() and not FORCE_RECOMPUTE_AE_INFERENCE:
        with open(cache_path, "rb") as f:
            payload = pickle.load(f)

        expected_keys = {"val_windows", "test_windows", "test_daily_df"}
        if expected_keys.issubset(payload.keys()):
            return payload

    csv_path = dataset_row["path"]

    df_dataset = pd.read_csv(csv_path)
    df_dataset["date"] = pd.to_datetime(df_dataset["date"])
    df_dataset = df_dataset.sort_values(["store_id", "date"]).reset_index(drop=True)

    df_dataset = prepare_pos_dataframe(df_dataset)
    df_dataset = ensure_pos_gt_columns(df_dataset)

    _, val, test, df_encoded = build_pos_ae_dataset_inference(
        df=df_dataset,
        feature_scalers=ae_payload["feature_scalers"],
        mappings=ae_payload["mappings"],
        features=ae_payload["features"],
        window_size=window_size,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
    )

    # Daily GT encoded con gli stessi mapping dell'AE, così gli store_id combaciano.
    test_daily_df = split_daily_df(
        df_encoded,
        split="test",
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
    )

    val_inputs = build_pos_ae_inputs(val)
    test_inputs = build_pos_ae_inputs(test)

    val_pred = ae_payload["model"].predict(val_inputs, verbose=0)
    test_pred = ae_payload["model"].predict(test_inputs, verbose=0)

    val_windows = make_pos_ae_reconstruction_results_df(
        val,
        val_pred,
        gt_cols=ae_payload["features"].get("ground_truth", POS_GT_COLS),
    )

    test_windows = make_pos_ae_reconstruction_results_df(
        test,
        test_pred,
        gt_cols=ae_payload["features"].get("ground_truth", POS_GT_COLS),
    )

    payload = {
        "val_windows": val_windows,
        "test_windows": test_windows,
        "test_daily_df": test_daily_df,
    }

    with open(cache_path, "wb") as f:
        pickle.dump(payload, f)

    return payload

## Esecuzione della griglia AE

Per ogni configurazione `window_size × latent_dim`:

1. si addestra/carica l'AE;
2. si ricostruiscono le finestre validation/test;
3. si calcola `ae_mae_score`;
4. si stima z-score su validation per store;
5. si costruiscono le finestre rilevate;
6. si valuta event-level sulle effect windows POS.

In [15]:
def ae_results_cache_is_usable():
    required_paths = [
        AE_RAW_RESULTS_PATH,
        AE_GT_EVAL_PATH,
        AE_DET_EVAL_PATH,
    ]

    if not all(path.exists() for path in required_paths):
        return False

    try:
        cached = pd.read_csv(AE_RAW_RESULTS_PATH)
    except (OSError, pd.errors.ParserError):
        return False

    required_cols = {
        "ae_window_size",
        "latent_dim",
        "score_col",
        "z_threshold",
        "min_consecutive",
        "gap_tolerance",
        "iou_threshold",
        "delay_type",
        "source_duration",
        "seed",
    }

    if cached.empty or not required_cols.issubset(cached.columns):
        return False

    expected_runs = {
        (
            int(window_size),
            int(latent_dim),
            str(dataset_row["delay_type"]),
            int(dataset_row["source_duration"]),
            int(dataset_row["seed"]),
        )
        for window_size, latent_dim in product(
            AE_WINDOW_VALUES,
            LATENT_DIM_VALUES,
        )
        for _, dataset_row in datasets_df.iterrows()
    }

    observed_runs = {
        (
            int(row.ae_window_size),
            int(row.latent_dim),
            str(row.delay_type),
            int(row.source_duration),
            int(row.seed),
        )
        for row in cached.itertuples(index=False)
    }

    if not expected_runs.issubset(observed_runs):
        return False

    return (
        set(cached["score_col"].dropna().astype(str)) == {AE_SCORE_COL}
        and np.allclose(cached["z_threshold"].dropna().astype(float), AE_Z_THRESHOLD)
        and np.allclose(cached["min_consecutive"].dropna().astype(float), AE_MIN_CONSECUTIVE)
        and np.allclose(cached["gap_tolerance"].dropna().astype(float), AE_GAP_TOLERANCE)
        and np.allclose(cached["iou_threshold"].dropna().astype(float), AE_IOU_THRESHOLD)
    )


if not FORCE_RECOMPUTE_RESULTS and ae_results_cache_is_usable():
    ae_results = pd.read_csv(AE_RAW_RESULTS_PATH)
    ae_gt_eval_all = pd.read_csv(AE_GT_EVAL_PATH)
    ae_det_eval_all = pd.read_csv(AE_DET_EVAL_PATH)

    print(f"Risultati AE caricati da: {AE_RAW_RESULTS_PATH}")

else:
    all_ae_rows = []
    all_ae_gt_eval = []
    all_ae_det_eval = []
    start_time = time.time()

    ae_grid = list(product(AE_WINDOW_VALUES, LATENT_DIM_VALUES))

    print(f"Configurazioni AE: {len(ae_grid)}")
    print(f"Datasets: {len(datasets_df)}")
    print("Detector AE reconstruction-based:")
    print({
        "score_col": AE_SCORE_COL,
        "z_threshold": AE_Z_THRESHOLD,
        "min_consecutive": AE_MIN_CONSECUTIVE,
        "gap_tolerance": AE_GAP_TOLERANCE,
        "iou_threshold": AE_IOU_THRESHOLD,
    })

    for window_size, latent_dim in tqdm(ae_grid, desc="Configurazioni AE"):
        ae_payload = train_or_load_ae(
            window_size=window_size,
            latent_dim=latent_dim,
        )

        for _, dataset_row in tqdm(
            datasets_df.iterrows(),
            total=len(datasets_df),
            desc=f"Datasets w={window_size}, latent={latent_dim}",
            leave=False,
        ):
            results_payload = compute_or_load_ae_reconstruction_results(
                window_size=window_size,
                latent_dim=latent_dim,
                ae_payload=ae_payload,
                dataset_row=dataset_row,
            )

            output = evaluate_ae_reconstruction_detector(
                val_windows=results_payload["val_windows"],
                test_windows=results_payload["test_windows"],
                test_daily_df=results_payload["test_daily_df"],
                score_col=AE_SCORE_COL,
                z_threshold=AE_Z_THRESHOLD,
                min_consecutive=AE_MIN_CONSECUTIVE,
                gap_tolerance=AE_GAP_TOLERANCE,
                iou_threshold=AE_IOU_THRESHOLD,
            )

            row = output["summary"].copy()
            row.update({
                "model": "AE_reconstruction",
                "detector_type": "reconstruction_zscore",
                "ae_window_size": window_size,
                "latent_dim": latent_dim,
                "score_col": AE_SCORE_COL,
                "z_threshold": AE_Z_THRESHOLD,
                "min_consecutive": AE_MIN_CONSECUTIVE,
                "gap_tolerance": AE_GAP_TOLERANCE,
                "iou_threshold": AE_IOU_THRESHOLD,
                "delay_type": dataset_row["delay_type"],
                "source_duration": int(dataset_row["source_duration"]),
                "seed": int(dataset_row["seed"]),
                "path": str(dataset_row["path"]),
            })
            all_ae_rows.append(row)

            gt_eval = output["gt_eval"].copy()
            if not gt_eval.empty:
                gt_eval["model"] = "AE_reconstruction"
                gt_eval["ae_window_size"] = window_size
                gt_eval["latent_dim"] = latent_dim
                gt_eval["delay_type"] = dataset_row["delay_type"]
                gt_eval["source_duration"] = int(dataset_row["source_duration"])
                gt_eval["seed"] = int(dataset_row["seed"])
                all_ae_gt_eval.append(gt_eval)

            det_eval = output["det_eval"].copy()
            if not det_eval.empty:
                det_eval["model"] = "AE_reconstruction"
                det_eval["ae_window_size"] = window_size
                det_eval["latent_dim"] = latent_dim
                det_eval["delay_type"] = dataset_row["delay_type"]
                det_eval["source_duration"] = int(dataset_row["source_duration"])
                det_eval["seed"] = int(dataset_row["seed"])
                all_ae_det_eval.append(det_eval)

    elapsed = time.time() - start_time
    print(f"AE grid completata in {elapsed / 60:.2f} minuti.")

    ae_results = pd.DataFrame(all_ae_rows)
    ae_gt_eval_all = (
        pd.concat(all_ae_gt_eval, ignore_index=True)
        if all_ae_gt_eval
        else pd.DataFrame()
    )
    ae_det_eval_all = (
        pd.concat(all_ae_det_eval, ignore_index=True)
        if all_ae_det_eval
        else pd.DataFrame()
    )

    ae_results.to_csv(AE_RAW_RESULTS_PATH, index=False)
    ae_gt_eval_all.to_csv(AE_GT_EVAL_PATH, index=False)
    ae_det_eval_all.to_csv(AE_DET_EVAL_PATH, index=False)

ae_results.head()

Risultati AE caricati da: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\ae_vs_lstm_comparison\tables\ae_reconstruction_grid_raw_results.csv


,n_gt_events,n_detected_events,tp,fp,fn,precision,recall,f1,mean_iou,mean_detection_delay,mean_det_offset_start,mean_det_offset_end,model,detector_type,ae_window_size,latent_dim,score_col,z_threshold,min_consecutive,gap_tolerance,iou_threshold,delay_type,source_duration,seed,path
0,50,40,5,35,45,0.125000,0.10,0.111111,0.688384,2.20,2.20,0.60,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,batch_backlog,1,42,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...
1,50,40,4,36,46,0.100000,0.08,0.088889,0.701389,2.25,2.25,0.25,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,batch_backlog,1,43,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...
2,50,41,5,36,45,0.121951,0.10,0.109890,0.672222,1.60,1.60,-0.40,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,batch_backlog,1,44,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...
3,50,40,4,36,46,0.100000,0.08,0.088889,0.701389,2.25,2.25,0.25,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,batch_backlog,1,45,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...
4,50,40,4,36,46,0.100000,0.08,0.088889,0.701389,2.25,2.25,0.25,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,batch_backlog,1,46,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...


## Salvataggio risultati AE

In [16]:
[
    AE_RAW_RESULTS_PATH,
    AE_GT_EVAL_PATH,
    AE_DET_EVAL_PATH,
]

[WindowsPath('C:/Users/ciok4/jupyter file/tesi/artifacts/results/pos_delay/ae_vs_lstm_comparison/tables/ae_reconstruction_grid_raw_results.csv'),
 WindowsPath('C:/Users/ciok4/jupyter file/tesi/artifacts/results/pos_delay/ae_vs_lstm_comparison/tables/ae_reconstruction_gt_eval.csv'),
 WindowsPath('C:/Users/ciok4/jupyter file/tesi/artifacts/results/pos_delay/ae_vs_lstm_comparison/tables/ae_reconstruction_det_eval.csv')]

## Aggregazione dei risultati AE

La tabella ordina le configurazioni AE per `F1` pooled. La configurazione in testa è la migliore osservata nella griglia e non rappresenta una selezione indipendente dal test.

In [17]:
ae_grid_overall = aggregate_results(
    ae_results,
    group_cols=[
        "model",
        "detector_type",
        "ae_window_size",
        "latent_dim",
        "score_col",
        "z_threshold",
        "min_consecutive",
        "gap_tolerance",
        "iou_threshold",
    ],
).sort_values(["f1_pooled", "mean_iou_mean"], ascending=[False, False])

ae_grid_overall.to_csv(AE_GRID_OVERALL_PATH, index=False)

ae_grid_overall.head(20)

,model,detector_type,ae_window_size,latent_dim,score_col,z_threshold,min_consecutive,gap_tolerance,iou_threshold,n_runs,n_gt_events,n_detected_events,tp,fp,fn,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,mean_iou_mean,mean_iou_std,det_offset_start_mean,det_offset_start_std,det_offset_end_mean,det_offset_end_std,precision_pooled,recall_pooled,f1_pooled
3,AE_reconstruction,reconstruction_zscore,5,16,ae_mae_score,3.5,2,1,0.2,25,1250,949,493,456,757,0.446117,0.228846,0.3944,0.270895,0.447796,0.231092,0.521909,0.052166,-2.678118,0.796602,-2.077371,0.404941,0.519494,0.3944,0.448386
7,AE_reconstruction,reconstruction_zscore,7,16,ae_mae_score,3.5,2,1,0.2,25,1250,705,125,580,1125,0.176847,0.036745,0.1000,0.022361,0.127751,0.027823,0.404009,0.043252,-4.192857,1.073312,-0.791429,0.828353,0.177305,0.1000,0.127877
6,AE_reconstruction,reconstruction_zscore,7,8,ae_mae_score,3.5,2,1,0.2,25,1250,1750,190,1560,1060,0.108571,0.007143,0.1520,0.010000,0.126667,0.008333,0.474190,0.027916,-3.467857,0.227752,0.428571,0.273745,0.108571,0.1520,0.126667
5,AE_reconstruction,reconstruction_zscore,7,4,ae_mae_score,3.5,2,1,0.2,25,1250,250,80,170,1170,0.320000,0.040825,0.0640,0.008165,0.106667,0.013608,0.388393,0.027338,-7.650000,0.714435,-1.700000,0.612372,0.320000,0.0640,0.106667
1,AE_reconstruction,reconstruction_zscore,5,4,ae_mae_score,3.5,2,1,0.2,25,1250,2228,183,2045,1067,0.082127,0.005179,0.1464,0.009522,0.105225,0.006707,0.642944,0.026270,1.645000,0.237088,-0.094286,0.249762,0.082136,0.1464,0.105233
10,AE_reconstruction,reconstruction_zscore,10,8,ae_mae_score,3.5,2,1,0.2,25,1250,583,88,495,1162,0.150507,0.027746,0.0704,0.014283,0.095920,0.018882,0.464795,0.029869,-9.152000,1.098529,0.362000,0.619691,0.150943,0.0704,0.096017
0,AE_reconstruction,reconstruction_zscore,5,2,ae_mae_score,3.5,2,1,0.2,25,1250,1003,108,895,1142,0.107634,0.011390,0.0864,0.009522,0.095853,0.010368,0.695288,0.010139,2.194000,0.182757,0.274000,0.247117,0.107677,0.0864,0.095872
9,AE_reconstruction,reconstruction_zscore,10,4,ae_mae_score,3.5,2,1,0.2,25,1250,750,95,655,1155,0.126667,0.025459,0.0760,0.015275,0.095000,0.019094,0.577601,0.042482,-4.509333,1.380771,0.360000,1.265899,0.126667,0.0760,0.095000
11,AE_reconstruction,reconstruction_zscore,10,16,ae_mae_score,3.5,2,1,0.2,25,1250,750,95,655,1155,0.126667,0.025459,0.0760,0.015275,0.095000,0.019094,0.521229,0.037707,-4.274000,1.314010,-0.640000,1.265899,0.126667,0.0760,0.095000
4,AE_reconstruction,reconstruction_zscore,7,2,ae_mae_score,3.5,2,1,0.2,25,1250,529,83,446,1167,0.156740,0.020936,0.0664,0.009522,0.093272,0.013056,0.426030,0.033491,-5.713333,1.476098,-1.286667,0.827088,0.156900,0.0664,0.093311


In [18]:
best_observed_ae_config = ae_grid_overall.iloc[0].to_dict()
best_observed_ae_config

{'model': 'AE_reconstruction',
 'detector_type': 'reconstruction_zscore',
 'ae_window_size': 5,
 'latent_dim': 16,
 'score_col': 'ae_mae_score',
 'z_threshold': 3.5,
 'min_consecutive': 2,
 'gap_tolerance': 1,
 'iou_threshold': 0.2,
 'n_runs': 25,
 'n_gt_events': 1250,
 'n_detected_events': 949,
 'tp': 493,
 'fp': 456,
 'fn': 757,
 'precision_mean': 0.44611687931170396,
 'precision_std': 0.2288458994560964,
 'recall_mean': 0.3944,
 'recall_std': 0.27089481353470013,
 'f1_mean': 0.4477963317761362,
 'f1_std': 0.23109160449269078,
 'mean_iou_mean': 0.521908900362925,
 'mean_iou_std': 0.052165664303042925,
 'det_offset_start_mean': -2.6781178332891273,
 'det_offset_start_std': 0.7966023824458469,
 'det_offset_end_mean': -2.0773705444979007,
 'det_offset_end_std': 0.40494093807387654,
 'precision_pooled': 0.5194942044257113,
 'recall_pooled': 0.3944,
 'f1_pooled': 0.4483856298317417}

In [19]:
best_observed_mask = (
    (ae_results["ae_window_size"] == best_observed_ae_config["ae_window_size"])
    & (ae_results["latent_dim"] == best_observed_ae_config["latent_dim"])
    & (ae_results["score_col"] == best_observed_ae_config["score_col"])
    & (ae_results["z_threshold"] == best_observed_ae_config["z_threshold"])
)

best_observed_ae_raw = ae_results[best_observed_mask].copy()

best_observed_ae_by_type = aggregate_results(
    best_observed_ae_raw,
    group_cols=["model", "delay_type"],
).sort_values("f1_pooled", ascending=False)

best_observed_ae_by_type

,model,delay_type,n_runs,n_gt_events,n_detected_events,tp,fp,fn,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,mean_iou_mean,mean_iou_std,det_offset_start_mean,det_offset_start_std,det_offset_end_mean,det_offset_end_std,precision_pooled,recall_pooled,f1_pooled
3,AE_reconstruction,settlement_freeze,5,250,273,182,91,68,0.665645,0.041115,0.728,0.070143,0.695225,0.054115,0.551333,0.004926,-2.683431,0.105830,-1.655094,0.132234,0.666667,0.728,0.695985
0,AE_reconstruction,batch_backlog,5,250,239,150,89,100,0.626594,0.055651,0.600,0.070711,0.612831,0.062955,0.516898,0.016001,-3.054165,0.147740,-1.969683,0.089883,0.627615,0.600,0.613497
4,AE_reconstruction,strong_delay,5,250,206,112,94,138,0.542311,0.051850,0.448,0.059330,0.490484,0.056561,0.480320,0.032735,-3.225556,0.298106,-2.264462,0.152076,0.543689,0.448,0.491228
2,AE_reconstruction,moderate_delay,5,250,131,37,94,213,0.279367,0.104939,0.148,0.057619,0.193315,0.074218,0.493318,0.031415,-2.762857,0.366394,-2.426667,0.113019,0.282443,0.148,0.194226
1,AE_reconstruction,mild_delay,5,250,100,12,88,238,0.116667,0.113064,0.048,0.046043,0.113347,0.029181,0.598187,0.089071,-0.988889,1.032975,-2.066667,0.901850,0.120000,0.048,0.068571


## Effetto congiunto di finestra e dimensione latente

Questa tabella mostra come cambiano le prestazioni AE al variare della scala temporale della finestra e della capacità del bottleneck.

In [20]:
ae_family_by_window_latent = (
    ae_grid_overall
    .groupby(["ae_window_size", "latent_dim"], as_index=False)
    .agg(
        n_configs=("f1_pooled", "count"),
        f1_median=("f1_pooled", "median"),
        f1_q75=("f1_pooled", lambda s: s.quantile(0.75)),
        f1_max_observed=("f1_pooled", "max"),
        precision_median=("precision_pooled", "median"),
        recall_median=("recall_pooled", "median"),
        mean_iou_median=("mean_iou_mean", "median"),
        mean_iou_max_observed=("mean_iou_mean", "max"),
    )
    .sort_values(["ae_window_size", "latent_dim"])
)

ae_family_by_window_latent.to_csv(AE_FAMILY_PATH, index=False)

ae_family_by_window_latent

,ae_window_size,latent_dim,n_configs,f1_median,f1_q75,f1_max_observed,precision_median,recall_median,mean_iou_median,mean_iou_max_observed
0,5,2,1,0.095872,0.095872,0.095872,0.107677,0.0864,0.695288,0.695288
1,5,4,1,0.105233,0.105233,0.105233,0.082136,0.1464,0.642944,0.642944
2,5,8,1,0.091429,0.091429,0.091429,0.160000,0.0640,0.247088,0.247088
3,5,16,1,0.448386,0.448386,0.448386,0.519494,0.3944,0.521909,0.521909
4,7,2,1,0.093311,0.093311,0.093311,0.156900,0.0664,0.426030,0.426030
5,7,4,1,0.106667,0.106667,0.106667,0.320000,0.0640,0.388393,0.388393
6,7,8,1,0.126667,0.126667,0.126667,0.108571,0.1520,0.474190,0.474190
7,7,16,1,0.127877,0.127877,0.127877,0.177305,0.1000,0.404009,0.404009
8,10,2,1,0.092728,0.092728,0.092728,0.118899,0.0760,0.569949,0.569949
9,10,4,1,0.095000,0.095000,0.095000,0.126667,0.0760,0.577601,0.577601


## Risultati LSTM forecasting-based

I risultati LSTM vengono riutilizzati dal notebook 02, che applica il detector POS a profili con la configurazione fissata. L'AE resta valutato con lo score di ricostruzione.

In [21]:
def load_lstm_results(
    path=LSTM_RESULTS_PATH,
    source_duration_filter=None,
    delay_types_filter=None,
):
    """Carica i risultati LSTM prodotti dal notebook 02."""

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            "Risultati LSTM non trovati: "
            f"{path}. Esegui prima 02-delay_type_sensitivity.py."
        )

    results = pd.read_csv(path)

    if source_duration_filter is not None:
        results = results[
            results["source_duration"].isin(source_duration_filter)
        ].copy()

    if delay_types_filter is not None:
        results = results[
            results["delay_type"].isin(delay_types_filter)
        ].copy()

    if results.empty:
        raise ValueError(
            "Il CSV LSTM non contiene risultati compatibili con i filtri "
            "dell'esperimento AE."
        )

    results["model"] = "LSTM_forecasting_profile"
    results["detector_type"] = "profile_detector"

    return results.reset_index(drop=True)


lstm_results = load_lstm_results(
    source_duration_filter=SOURCE_DURATION_FILTER,
    delay_types_filter=DELAY_TYPES_FILTER,
)

print(f"Risultati LSTM caricati da: {LSTM_RESULTS_PATH}")
lstm_results.head()

Risultati LSTM caricati da: C:\Users\ciok4\jupyter file\tesi\artifacts\results\pos_delay\lstm_sensitivity\pos_delay_sensitivity_best_detector_results.csv


,n_gt_events,n_detected_events,tp,fp,fn,precision,recall,f1,mean_iou,mean_detection_delay,mean_det_offset_start,mean_det_offset_end,delay_type,source_duration,seed,path,score_col,profile_window_size,z_threshold,min_consecutive,gap_tolerance,detected_window_mode,iou_threshold,model,detector_type
0,50,68,50,18,0,0.735294,1.00,0.847458,0.477820,-5.680000,-5.680000,1.980000,batch_backlog,1,42,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,pos_cos,7,3.5,2,1,profile_windows_union,0.2,LSTM_forecasting_profile,profile_detector
1,50,68,47,21,3,0.691176,0.94,0.796610,0.450905,-6.021277,-6.021277,1.744681,batch_backlog,1,43,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,pos_cos,7,3.5,2,1,profile_windows_union,0.2,LSTM_forecasting_profile,profile_detector
2,50,69,50,19,0,0.724638,1.00,0.840336,0.431884,-6.520000,-6.520000,1.880000,batch_backlog,1,44,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,pos_cos,7,3.5,2,1,profile_windows_union,0.2,LSTM_forecasting_profile,profile_detector
3,50,70,50,20,0,0.714286,1.00,0.833333,0.450422,-6.180000,-6.180000,1.620000,batch_backlog,1,45,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,pos_cos,7,3.5,2,1,profile_windows_union,0.2,LSTM_forecasting_profile,profile_detector
4,50,66,48,18,2,0.727273,0.96,0.827586,0.437561,-6.437500,-6.437500,1.958333,batch_backlog,1,46,C:\Users\ciok4\jupyter file\tesi\artifacts\dat...,pos_cos,7,3.5,2,1,profile_windows_union,0.2,LSTM_forecasting_profile,profile_detector


## Aggregazione risultati LSTM

In [22]:
lstm_overall = aggregate_results(
    lstm_results,
    group_cols=["model", "detector_type"],
)

lstm_by_type = aggregate_results(
    lstm_results,
    group_cols=["model", "detector_type", "delay_type"],
).sort_values("f1_pooled", ascending=False)

lstm_overall

,model,detector_type,n_runs,n_gt_events,n_detected_events,tp,fp,fn,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,mean_iou_mean,mean_iou_std,det_offset_start_mean,det_offset_start_std,det_offset_end_mean,det_offset_end_std,precision_pooled,recall_pooled,f1_pooled
0,LSTM_forecasting_profile,profile_detector,25,1250,1527,1022,505,228,0.654482,0.0971,0.8176,0.227326,0.722805,0.154699,0.474865,0.051353,-5.421983,0.81951,1.610437,1.580901,0.669286,0.8176,0.736046


In [23]:
lstm_by_type

,model,detector_type,delay_type,n_runs,n_gt_events,n_detected_events,tp,fp,fn,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,mean_iou_mean,mean_iou_std,det_offset_start_mean,det_offset_start_std,det_offset_end_mean,det_offset_end_std,precision_pooled,recall_pooled,f1_pooled
3,LSTM_forecasting_profile,profile_detector,settlement_freeze,5,250,340,249,91,1,0.732480,0.012695,0.996,0.008944,0.844116,0.010462,0.403299,0.009424,-5.739020,0.326429,4.441306,0.293627,0.732353,0.996,0.844068
0,LSTM_forecasting_profile,profile_detector,batch_backlog,5,250,341,245,96,5,0.718533,0.017037,0.980,0.028284,0.829065,0.019615,0.449718,0.017730,-6.167755,0.337768,1.836603,0.152177,0.718475,0.980,0.829103
4,LSTM_forecasting_profile,profile_detector,strong_delay,5,250,339,240,99,10,0.708080,0.018467,0.960,0.028284,0.814924,0.020200,0.477580,0.016856,-5.842637,0.415252,1.161831,0.085621,0.707965,0.960,0.814941
2,LSTM_forecasting_profile,profile_detector,moderate_delay,5,250,290,182,108,68,0.628131,0.030489,0.728,0.022804,0.674290,0.026534,0.508884,0.029259,-5.209252,0.598025,0.546463,0.234292,0.627586,0.728,0.674074
1,LSTM_forecasting_profile,profile_detector,mild_delay,5,250,217,106,111,144,0.485187,0.042789,0.424,0.077974,0.451628,0.063284,0.534842,0.033681,-4.151248,0.402236,0.065980,0.307223,0.488479,0.424,0.453961


## Confronto tra LSTM e Autoencoder

La tabella compatta riporta l'LSTM e tutte le configurazioni AE della griglia. La configurazione AE migliore è identificata solo dopo la valutazione dei risultati.

In [24]:
lstm_compact = lstm_overall.copy()

lstm_compact["Modello"] = "LSTM"
lstm_compact["w_AE"] = "--"
lstm_compact["d_lat"] = "--"

lstm_compact = lstm_compact.rename(
    columns={
        "n_detected_events": "Eventi rilevati",
        "precision_pooled": "Precision",
        "recall_pooled": "Recall",
        "f1_pooled": "F1",
        "mean_iou_mean": "IoU media",
    }
)

# =========================
# AE ROWS
# =========================

ae_compact = ae_grid_overall.copy()

ae_compact["Modello"] = "AE"
ae_compact["w_AE"] = ae_compact["ae_window_size"].astype(int)
ae_compact["d_lat"] = ae_compact["latent_dim"].astype(int)

ae_compact = ae_compact.rename(
    columns={
        "n_detected_events": "Eventi rilevati",
        "precision_pooled": "Precision",
        "recall_pooled": "Recall",
        "f1_pooled": "F1",
        "mean_iou_mean": "IoU media",
    }
)

# =========================
# FINAL TABLE
# =========================

compact_model_comparison = pd.concat(
    [
        lstm_compact,
        ae_compact,
    ],
    ignore_index=True,
    sort=False,
)

compact_model_comparison = compact_model_comparison[
    [
        "Modello",
        "w_AE",
        "d_lat",
        "Eventi rilevati",
        "Precision",
        "Recall",
        "F1",
        "IoU media",
    ]
].copy()

# LSTM prima, AE ordinate per F1 decrescente
compact_model_comparison["_sort_model"] = np.where(
    compact_model_comparison["Modello"] == "LSTM",
    0,
    1,
)

compact_model_comparison = (
    compact_model_comparison
    .sort_values(
        ["_sort_model", "F1"],
        ascending=[True, False],
    )
    .drop(columns="_sort_model")
    .reset_index(drop=True)
)

# Arrotondamento per visualizzazione
compact_model_comparison_display = compact_model_comparison.copy()

for col in ["Precision", "Recall", "F1", "IoU media"]:
    compact_model_comparison_display[col] = (
        compact_model_comparison_display[col]
        .astype(float)
        .round(3)
    )

compact_model_comparison_display["Eventi rilevati"] = (
    compact_model_comparison_display["Eventi rilevati"]
    .astype(int)
)

display(compact_model_comparison_display)

# =========================
# SAVE
# =========================

compact_table_path = COMPACT_TABLE_PATH

compact_model_comparison_display.to_csv(
    compact_table_path,
    index=False,
)

compact_table_path

,Modello,w_AE,d_lat,Eventi rilevati,Precision,Recall,F1,IoU media
0,LSTM,--,--,1527,0.669,0.818,0.736,0.475
1,AE,5,16,949,0.519,0.394,0.448,0.522
2,AE,7,16,705,0.177,0.100,0.128,0.404
3,AE,7,8,1750,0.109,0.152,0.127,0.474
4,AE,7,4,250,0.320,0.064,0.107,0.388
5,AE,5,4,2228,0.082,0.146,0.105,0.643
6,AE,10,8,583,0.151,0.070,0.096,0.465
7,AE,5,2,1003,0.108,0.086,0.096,0.695
8,AE,10,4,750,0.127,0.076,0.095,0.578
9,AE,10,16,750,0.127,0.076,0.095,0.521


WindowsPath('C:/Users/ciok4/jupyter file/tesi/artifacts/results/pos_delay/ae_vs_lstm_comparison/tables/compact_lstm_vs_all_ae_configs.csv')